# Coding SSM Forward: GPU

Implement the forward pass of a linear state space model using recurrence and
convolution, and compare measured runtime. Use a CUDA GPU (for example a free Colab GPU), or Apple Silicon MPS. The notebook falls back to CPU for correctness checks if no supported GPU is available; label such results as CPU and do not claim a GPU comparison. Copy your implementations from the CPU notebook for the first part.

Install Python 3.10+ with PyTorch 2.0+ and Matplotlib (see the accompanying README).
Colab already includes these packages. All measurements use float32 inference.
The first sanity check will fail until you complete the TODO functions.

## Imports

In [ ]:
import time
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    print("No CUDA/MPS GPU available: CPU correctness fallback.")

print("PyTorch:", torch.__version__, "device:", device,
      "CPU threads:", torch.get_num_threads())
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))

# SSM Update Rule

We consider a linear recurrent model. For each sequence in the batch,

$$h_{t+1}=W h_t+U x_t+b,\qquad t=0,\ldots,T-1,\qquad h_0=0.$$

The stored input `x` has shape `(N, T, D)`. Each individual $x_t\in\mathbb R^D$
and $h_t\in\mathbb R^H$; $W\in\mathbb R^{H\times H}$,
$U\in\mathbb R^{H\times D}$, and $b\in\mathbb R^H$ are shared across the batch.
Return an array `h_all` of shape `(N, T, H)` with `h_all[:, t, :]` storing $h_{t+1}$.

Use the input tensor's dtype and device for all intermediate tensors.

Implement `unrolled_ssm_forward`: unroll the recurrence in time and
return every hidden state. Do not add a nonlinear activation.

In [ ]:
def unrolled_ssm_forward(W, U, b, x):
    """Return (N,T,H), with output[:,t,:] equal to h_{t+1}; h_0=0."""
    # TODO: Unroll the recurrence, preserving dtype and device.
    raise NotImplementedError("Complete this function before running the checks.")

# Convolution Based Implementation

Let $s_t=U x_t+b$. Unrolling the update gives
$$h_{t+1}=\sum_{j=0}^{t}W^{t-j}s_j.$$
Implement this causal convolution with `torch.nn.functional.conv1d` (the functional
form of `nn.Conv1d`). PyTorch's `conv1d` computes cross-correlation: decide which
order the matrix powers need in the kernel, and how to pad/crop for causality.

Implement `make_conv_kernel(W, T)`, returning shape `(H, H, T)`, and
`conv_ssm_forward(W, U, b, x)`, returning shape `(N, T, H)`.
You may assume positive $T$ is a power of two and use divide-and-conquer to build
all matrix powers. Include the input bias at **every** time step.

In [ ]:
def make_conv_kernel(W, T):
    """Build the (H,H,T) kernel for causal conv1d; T is a power of two."""
    # TODO: Construct and order the matrix powers.
    raise NotImplementedError("Complete this function before running the checks.")

In [ ]:
def conv_ssm_forward(W, U, b, x):
    """Compute all h_1,...,h_T by causal dense conv1d."""
    # TODO: Project the input, add bias, construct the kernel, and apply conv1d.
    raise NotImplementedError("Complete this function before running the checks.")

# Sanity Check
We can compare the outputs of the two implementations to check if they are consistent.

In [ ]:
def sanity_check():
    torch.manual_seed(0)
    for T in (1, 2, 8):
        N, H, D = 2, 4, 3
        W = torch.randn(H, H, device=device) * 0.1
        U = torch.randn(H, D, device=device) * 0.1
        b = torch.randn(H, device=device) * 0.1
        x = torch.randn(N, T, D, device=device)
        h_recurrent = unrolled_ssm_forward(W, U, b, x)
        h_conv = conv_ssm_forward(W, U, b, x)
        assert h_recurrent.shape == (N, T, H)
        torch.testing.assert_close(h_recurrent, h_conv, rtol=1e-4, atol=1e-5)
        print(f"T={T}: maximum absolute difference = "
              f"{(h_recurrent - h_conv).abs().max().item():.3g}")


sanity_check()

# Runtime Comparison

The timer includes input projection, bias, kernel construction, and forward
evaluation. It warms up each method and synchronizes CUDA/MPS before and after
the timed region, so asynchronous device execution is included. Use the same
dimensions and thread count when comparing against the CPU notebook. For a
matched CPU/GPU comparison, rerun both with `H_values = [4, 16, 32]`,
`T_values = [8, 32, 128]`, `N=8`, and `D=16`. Record the GPU model, backend,
PyTorch version, dimensions, and actual timings. No universal speedup is expected.
Start with the bounded grid below; dense kernel storage grows as $TH^2$.

In [ ]:
def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def measure_runtime(method_fn, W, U, b, x, warmup=1, repeats=3):
    """Wall time including kernel construction and input projection, in seconds."""
    with torch.inference_mode():
        for _ in range(warmup):
            method_fn(W, U, b, x)
        synchronize(x.device)
        start = time.perf_counter()
        for _ in range(repeats):
            method_fn(W, U, b, x)
        synchronize(x.device)
        return (time.perf_counter() - start) / repeats


def run_benchmark(recurrent_fn, convolution_fn, H_values, T_values, N=8, D=16,
                  diagonal=False):
    torch.manual_seed(0)
    results = {}
    for H in H_values:
        U = torch.randn(H, D, device=device) * 0.1
        b = torch.randn(H, device=device) * 0.1
        recurrent_times, convolution_times = [], []
        for T in T_values:
            if diagonal:
                W = torch.diag(torch.randn(H, device=device) * 0.1)
            else:
                # Keep recurrent powers from exploding as H changes.
                W = torch.randn(H, H, device=device) * (0.1 / H**0.5)
            x = torch.randn(N, T, D, device=device)
            with torch.inference_mode():
                torch.testing.assert_close(recurrent_fn(W, U, b, x),
                                           convolution_fn(W, U, b, x),
                                           rtol=1e-4, atol=1e-5)
            recurrent_times.append(measure_runtime(recurrent_fn, W, U, b, x))
            convolution_times.append(measure_runtime(convolution_fn, W, U, b, x))
        results[H] = (recurrent_times, convolution_times)
        print(f"H={H}: recurrent={recurrent_times}, convolution={convolution_times}")
    for H, (recurrent_times, convolution_times) in results.items():
        plt.figure(figsize=(6, 4))
        plt.plot(T_values, recurrent_times, "o-", label="Recurrent")
        plt.plot(T_values, convolution_times, "s-", label="Convolution")
        plt.title(f"{device}: N={N}, D={D}, H={H}, diagonal={diagonal}")
        plt.xlabel("Time steps T")
        plt.ylabel("Mean wall time (seconds)")
        plt.yscale("log")
        plt.grid(True)
        plt.legend()
        plt.show()
    return results


# Start with this small grid. Extend it only if your runtime and memory allow.
H_values = [4, 16, 64]
T_values = [8, 32, 128]
results = run_benchmark(unrolled_ssm_forward, conv_ssm_forward, H_values, T_values)

### Question 6

Compare CPU and GPU results at matched dimensions. How do $T$ and $H$ affect
the measurements? Explain the effects of parallelism, device overhead, kernel
construction, and arithmetic work. If using CPU fallback, state that a GPU
comparison could not be measured and explain the predicted trade-off.

# Introducing Structure: Diagonal Weight Matrices

Now constrain $W$ to be diagonal. Build a `(H, 1, T)` kernel and use depthwise
`conv1d` with `groups=H`. Ensure the code supports distinct diagonal entries;
an identity matrix alone is not a sufficient correctness check.

In [ ]:
def make_diag_depthwise_kernel(W, T):
    """Build an (H,1,T) causal conv1d kernel; W is diagonal."""
    # TODO: Construct the correctly ordered elementwise powers.
    raise NotImplementedError("Complete this function before running the checks.")

In [ ]:
def diag_conv_ssm_forward(W, U, b, x):
    """Compute all states by depthwise conv1d; W is diagonal."""
    # TODO: Project the input, add bias, and apply a causal grouped convolution.
    raise NotImplementedError("Complete this function before running the checks.")

# Optimizing the recurrent implementation

Similarly, we can also optimize the recurrent implementation by using the diagonal nature of the $W$ matrix.

In [ ]:
def diag_unrolled_ssm_forward(W, U, b, x):
    """Compute all states by elementwise recurrence; W is diagonal."""
    # TODO: Use the diagonal to avoid a dense recurrent matrix multiply.
    raise NotImplementedError("Complete this function before running the checks.")

# Sanity Check

We can compare the outputs of the two implementations to check if they are consistent similar to above.

In [ ]:
def diag_sanity_check():
    torch.manual_seed(1)
    for T in (1, 2, 8):
        N, H, D = 2, 4, 3
        # Distinct, nonzero diagonal entries catch channel-order mistakes.
        W = torch.diag(torch.tensor([0.1, -0.2, 0.3, -0.4], device=device))
        U = torch.randn(H, D, device=device) * 0.1
        b = torch.randn(H, device=device) * 0.1
        x = torch.randn(N, T, D, device=device)
        expected = unrolled_ssm_forward(W, U, b, x)
        recurrent = diag_unrolled_ssm_forward(W, U, b, x)
        convolution = diag_conv_ssm_forward(W, U, b, x)
        torch.testing.assert_close(recurrent, expected, rtol=1e-4, atol=1e-5)
        torch.testing.assert_close(convolution, expected, rtol=1e-4, atol=1e-5)
        print(f"T={T}: maximum absolute difference = "
              f"{(recurrent - convolution).abs().max().item():.3g}")


diag_sanity_check()

# Measure Runtime with Diagonal Weights

Use the same batch size, dimensions, dtype, and timing rules as the dense case.
Compare the optimized diagonal recurrence against depthwise convolution. You may
extend the grid if resources permit, but record any changed dimensions.

In [ ]:
diag_results = run_benchmark(diag_unrolled_ssm_forward,
                             diag_conv_ssm_forward,
                             H_values, T_values, N=8, D=16, diagonal=True)

### Question 7

How do diagonal weights change arithmetic work and kernel storage? Compare
the optimized recurrence and depthwise convolution at matched dimensions and
explain the measured results. Is the convolution always faster? Support your
answer with evidence and distinguish direct convolution from FFT methods.